# Getting Started with SpatialEconometrics.jl

A hands-on tutorial covering the complete spatial econometrics workflow:

1. Creating spatial weight matrices
2. Simulating spatial data
3. Running spatial diagnostics (Moran's I, LM tests)
4. Fitting SAR and SDM models
5. Computing spatial effects (direct, indirect, total)
6. Comparing with the Spatial Error Model (SEM)

> **Note**: First execution of each cell takes longer due to Julia's JIT compilation. Subsequent calls are fast.

## 1. Setup

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

using SpatialEconometrics
using Random
using LinearAlgebra
using SparseArrays
using Statistics

## 2. Spatial Weight Matrices

A spatial weight matrix **W** encodes the neighbourhood structure. Each entry $w_{ij}$ measures
the "spatial influence" of unit $j$ on unit $i$. Common constructions:

- **Rook contiguity**: units sharing an edge (4 neighbors on a grid)
- **Queen contiguity**: units sharing an edge or corner (8 neighbors on a grid)

Both are **row-standardized** so each row sums to 1.

In [ ]:
W_rook  = make_rook_weights(15)
W_queen = make_queen_weights(15)
N = size(W_rook, 1)

println("Rook weights:  N=$N, nnz=$(nnz(W_rook)), avg neighbors=$(round(nnz(W_rook)/N; digits=1))")
println("Queen weights: N=$N, nnz=$(nnz(W_queen)), avg neighbors=$(round(nnz(W_queen)/N; digits=1))")
println("Row sums (should all be 1.0): ", unique(round.(vec(sum(W_rook; dims=2)); digits=10)))

## 3. Simulating Spatial Data

We generate data from a **Spatial Durbin Model (SDM)** DGP:

$$y = (I - \rho W)^{-1} (\beta_0 + X\beta + WX\theta + \varepsilon)$$

| Parameter | Value | Meaning |
|-----------|-------|---------|
| $\rho$ | 0.55 | Spatial autoregressive parameter |
| $\beta$ | [1.5, -1.0] | Direct covariate effects |
| $\theta$ | [0.8, -0.5] | Spatial lag covariate effects |
| $\sigma$ | 0.5 | Error standard deviation |

In [ ]:
rng = MersenneTwister(42)
W = W_rook

X = randn(rng, N, 2)
β_true = [2.0, 1.5, -1.0]
θ_true = [0.8, -0.5]
ρ_true = 0.55
σ_true = 0.50

y = (I - ρ_true * W) \ (β_true[1] .+ X * β_true[2:3] .+ (W * X) * θ_true .+ σ_true .* randn(rng, N))

println("y: mean=$(round(mean(y); digits=2)), std=$(round(std(y); digits=2))")
println("   range=[$(round(minimum(y); digits=2)), $(round(maximum(y); digits=2))]")

## 4. Spatial Diagnostics

Before fitting a spatial model, test for spatial autocorrelation and determine which specification is appropriate.

### 4.1 Global Moran's I

Tests whether $y$ exhibits spatial clustering. Under the null of no spatial autocorrelation,
$I \approx -1/(N-1) \approx 0$.

In [ ]:
mr = moran_global(y, W)
println(mr)
println("\nI = $(round(mr.I_stat; digits=4)) >> 0, p ≈ 0 → strong positive spatial autocorrelation")

### 4.2 Geary's C

Complementary to Moran's I. Under no autocorrelation $C = 1$; positive autocorrelation → $C < 1$.

In [ ]:
gc = geary_c(y, W)
println(gc)

### 4.3 LM Specification Tests (Anselin 1988)

The **Lagrange Multiplier tests** help choose between:
- **SAR** (spatial lag): spatial dependence in $y$
- **SEM** (spatial error): spatial dependence in $\varepsilon$

**Decision rule** using the robust variants:

| Robust LM-Lag | Robust LM-Error | → Model |
|:---:|:---:|:---:|
| Significant | Not significant | **SAR / SDM** |
| Not significant | Significant | **SEM** |
| Both significant | Both significant | **SARAR (GS2SLS)** |

In [ ]:
X_ols = hcat(ones(N), X)
lms = lm_tests(y, X_ols, W)

println(lms.lag)
println(lms.error)
println(lms.robust_lag)
println(lms.robust_error)
println("\n→ Robust LM-Lag significant, Robust LM-Error not → SAR/SDM family")

### 4.4 Local Moran's I (LISA)

Identifies local clusters: High-High (HH), Low-Low (LL), High-Low (HL), Low-High (LH).

In [ ]:
lisa = moran_local(y, W; n_perm=499, rng=MersenneTwister(99))
println(lisa)

sig = lisa.pseudo_p .< 0.05
println("Significant local clusters: $(sum(sig)) / $N")
for ct in [:HH, :LL, :HL, :LH]
    n = count(lisa.cluster_type .== ct)
    n > 0 && println("  $ct: $n")
end

## 5. Fitting Spatial Models

### 5.1 SAR Model

$y = \rho W y + X\beta + \varepsilon$

Estimated by concentrated ML with Chebyshev log-determinant approximation.

In [ ]:

m_sar = fit_spatial(SAR(), W, X, y;
    method = ChebyshevApprox(order=12),
    varnames = ["income", "density"])
println(m_sar)

### 5.2 SDM Model (correct specification)

$y = \rho W y + X\beta + WX\theta + \varepsilon$

In [ ]:
m_sdm = fit_spatial(SDM(), W, X, y;
    method = ChebyshevApprox(order=12),
    varnames = ["income", "density"])
println(m_sdm)

println("\nParameter recovery:")
println("  ρ:  true=$ρ_true  →  est=$(round(m_sdm.rho; digits=4))")
println("  β₁: true=$(β_true[2])  →  est=$(round(m_sdm.coef[2]; digits=4))")
println("  θ₁: true=$(θ_true[1])  →  est=$(round(m_sdm.coef[4]; digits=4))")

## 6. Spatial Effects (LeSage & Pace 2009)

A change in $x_k$ at location $i$ affects $y$ everywhere through the spatial multiplier
$(I - \rho W)^{-1}$:

| Effect | Definition | Interpretation |
|--------|-----------|----------------|
| **Direct** | Avg diagonal of $(I-\rho W)^{-1}(\beta_k I + \theta_k W)$ | Own-region effect |
| **Total** | $(\beta_k + \theta_k)/(1 - \rho)$ | Full equilibrium effect |
| **Indirect** | Total − Direct | Spillover to neighbors |

Standard errors via the **delta method** (ForwardDiff).

In [ ]:
effects = spatial_effects(m_sdm)
println(effects)

In [ ]:
# Compare with theoretical values
println("Comparison with true total effects:")
for k in 1:2
    total_true = (β_true[k+1] + θ_true[k]) / (1 - ρ_true)
    println("  $(effects.varnames[k]): true=$(round(total_true; digits=3)), est=$(round(effects.total[k]; digits=3))")
end

## 7. Comparing with SEM

The **Spatial Error Model**: $y = X\beta + u$, $u = \lambda W u + \varepsilon$

In SEM, spatial dependence is in the errors, not in $y$. No indirect effects — only direct
effects equal to $\beta$.

Since our DGP is SDM, this is **misspecified** — useful for comparison.

In [ ]:
m_sem = fit_spatial_error(W, X, y; varnames=["income", "density"])
println(m_sem)

In [ ]:
println("Model Comparison")
println("═══════════════════════════════════════════")
println("             SAR        SDM        SEM")
println("───────────────────────────────────────────")
println("ρ/λ       $(round(m_sar.rho;digits=4))    $(round(m_sdm.rho;digits=4))    $(round(m_sem.lambda;digits=4))")
println("σ²        $(round(m_sar.sigma2;digits=4))    $(round(m_sdm.sigma2;digits=4))    $(round(m_sem.sigma2;digits=4))")
println("Log-lik   $(round(m_sar.loglik;digits=1))   $(round(m_sdm.loglik;digits=1))   $(round(m_sem.loglik;digits=1))")
println("═══════════════════════════════════════════")
println("\n→ SDM has best log-likelihood (correct specification)")

## 8. Formula Interface

R-style formulas with spatial extensions: `slag()`, `fe()`, `endo()`.

In [ ]:
tbl = (y = y, income = X[:, 1], density = X[:, 2])

m_formula = fit_spatial(
    @formula(y ~ income + density + slag(income + density)),
    tbl, W)

println("Formula SDM ρ̂ = $(round(m_formula.rho; digits=6))")
println("Matrix  SDM ρ̂ = $(round(m_sdm.rho; digits=6))")
println("Identical: $(m_formula.rho == m_sdm.rho)")

## Summary

| Step | Function | Result |
|------|----------|--------|
| Test for autocorrelation | `moran_global` | I > 0, p < 0.001 |
| Choose SAR vs SEM | `lm_tests` | Robust LM-Lag significant → SAR family |
| Fit SAR | `fit_spatial(SAR(), ...)` | ρ̂ recovered |
| Fit SDM | `fit_spatial(SDM(), ...)` | β, θ, ρ all recovered |
| Spatial spillovers | `spatial_effects(m)` | Direct + indirect decomposition |
| Compare SEM | `fit_spatial_error(...)` | Lower log-lik (misspecified) |

**Next**: See `02_spatial_panels.ipynb` for panel data and `03_gwr_gravity.ipynb` for GWR and gravity models.